# Phase 45 — TensorFlow Retraining with multilingual-e5-small Features

This notebook retrains the TensorFlow job-fit scorer with `intfloat/multilingual-e5-small`-derived features. It preserves the Phase 25 model-core contract and writes a new artifact namespace only.


## Step 45.1 — Migration notebook setup

### Purpose
Create a notebook-first migration path for multilingual-E5-small retraining without mutating Phase 25 artifacts.

### Required input
- `TODOS.md` Phase 45 scope.
- `GAP_MODEL_TRAINING.md` model-owned output boundary.
- `REQUIREMENT.md` TensorFlow/custom-loop/TensorBoard requirements.

### Action
Load the executable Phase 45 verification module and reserve `artifacts/phase_45_multilingual_e5_small_training_delivery/`.

### Expected output
A runnable notebook with English documentation before code and a Phase 45 artifact namespace.

### Verification
The verification script checks notebook existence, required evidence paths, and namespace isolation.


## Step 45.2 — Frozen input reuse

### Purpose
Use the same frozen dataset, labels, splits, pair IDs, human validation labels, ATS benchmark, and backend candidate fixtures as Phase 25 unless separately approved.

### Required input
- `artifacts/phase_25_tensorflow_training_delivery/tensorflow_feature_config.json`.
- `artifacts/phase_25_tensorflow_training_delivery/label_manifest.json`.
- `artifacts/phase_25_tensorflow_training_delivery/dataset_manifest.json`.
- `reports/phase_21_backend_candidate_reranking.json`.
- `artifacts/phase_44_embedding_compatibility_audit/multilingual_e5_small_pair_features.json`.

### Action
Validate required source artifacts and reuse Phase 44 rows keyed by frozen Phase 25 pair IDs and splits.

### Expected output
A source-hash record inside `tensorflow_feature_config.json`.

### Verification
The verification script fails if required frozen source artifacts are missing.


## Step 45.3 — Approved multilingual-E5-small feature generation

### Purpose
Rebuild the approved six-feature TensorFlow matrix with multilingual-E5-small `e5_cosine`.

### Required input
- Phase 44 `multilingual_e5_small_cosine`.
- Frozen numeric features: `skill_overlap`, `requirement_coverage`, `role_match`, `experience_match`, and `experience_gap_years_clipped`.

### Action
Write `tensorflow_training_features_v1.npz` with `X_raw`, `X_scaled`, `y`, `pair_id`, `split`, and `feature_names`.

### Expected output
The approved feature order stays `e5_cosine`, `skill_overlap`, `requirement_coverage`, `role_match`, `experience_match`, `experience_gap_years_clipped`.

### Verification
The verification script checks row count, split count, target bounds, and feature order.


## Step 45.4 — Train-split normalization refresh

### Purpose
Replace Phase 25 normalization stats with train-split stats computed from multilingual-E5-small features.

### Required input
The Phase 45 raw six-feature matrix and train split indicator.

### Action
Compute mean/std from train rows only and write `tensorflow_feature_config.json` with embedding model metadata.

### Expected output
A feature config that records `intfloat/multilingual-e5-small`, dimension `384`, `query:`/`passage:` prefixes, normalized embeddings, and new normalization stats.

### Verification
The verification script fails if normalization stats or embedding metadata are missing.


## Step 45.5 — TensorFlow GradientTape training

### Purpose
Train a new TensorFlow Functional API scorer with custom components and a manual `tf.GradientTape` loop.

### Required input
- Phase 45 normalized feature matrix.
- `CosineInteractionLayer`, `WeightedHuberLoss`, and `ProductionGateCallback`.

### Action
Train the model without `model.fit()`, save history, predictions, a `.keras` candidate, and clean reload smoke evidence.

### Expected output
`gradient_tape_training_history.csv`, `gradient_tape_predictions_v1.npz`, `gradient_tape_trained_candidate.keras`, and `training_evaluation.json`.

### Verification
The script fails if GradientTape is not used, `model.fit()` is used, predictions are non-finite/out of bounds, or clean reload fails.


## Step 45.6 — Baseline comparison

### Purpose
Compare the multilingual-E5-small TensorFlow candidate against required baselines and previous selected scorers.

### Required input
- Constant train mean/median baselines.
- Skill-only and cosine-only regressions.
- Phase 25 E5-base TensorFlow predictions.
- Previous selected scorer metrics from Phase 25 evidence.

### Action
Compute validation/test metrics and non-regression checks.

### Expected output
`baseline_comparison.json` and `selection_decision.json`.

### Verification
Selection is only allowed when quality and non-regression gates pass; otherwise Phase 25 remains selected.


## Step 45.7 — Quality, ranking, and slice metrics

### Purpose
Record regression, ranking, and slice metrics needed for staging review.

### Required input
Phase 45 predictions, targets, pair metadata, and frozen splits.

### Action
Compute MAE, RMSE, R², Spearman, score-band agreement, high-fit recall, NDCG@5, NDCG@10, MAP@10, and slices by language, role, experience-related pair metadata, and pair type.

### Expected output
Quality metrics in `training_evaluation.json`.

### Verification
The verification script requires the core metric set and ranking metric set.


## Step 45.8 — Indonesian and mixed-language behavior

### Purpose
Check dedicated Indonesian and mixed-language examples for practical staging behavior.

### Required input
Phase 45 pair metadata with `language` labels and predictions.

### Action
Record representative ID, MIXED, EN, and UNKNOWN examples without raw CV text.

### Expected output
`indonesian_behavior_examples` in `training_evaluation.json`.

### Verification
The verification script fails if no Indonesian or mixed-language behavior evidence is recorded when slices exist.


## Step 45.9 — TensorBoard evidence

### Purpose
Record bounded TensorBoard evidence for manual GradientTape training.

### Required input
Training history, final metrics, predictions, and raw features.

### Action
Write scalar and histogram summaries under `artifacts/tensorboard/phase_45_multilingual_e5_small_training_delivery/`.

### Expected output
`tensorboard_monitoring_manifest.json` with SHA-256 and byte size for event files.

### Verification
The verification script requires at least one event file with hash and byte-size metadata.


## Step 45.10 — Select or reject model

### Purpose
Select the new model only if predefined staging thresholds pass.

### Required input
Baseline comparison, quality metrics, slice evidence, TensorBoard manifest, and clean reload smoke.

### Action
Write a selection or rejection decision. Do not update Model API defaults or mix Phase 25 calibration with Phase 45 model files.

### Expected output
`selection_decision.json` and `reports/phase_45_multilingual_e5_small_training_delivery.json`.

### Verification
The verification script records either `select_for_staging_shadow_validation` or `reject_keep_e5_base_phase25_selected`.


In [ ]:
from scripts.verify_phase_45_multilingual_e5_small_training_delivery import write_all

report = write_all()
report["status"], report["selection_decision"], report["blockers"]
